# nn.Module 与 nn.Parameter

**核心机制**

- nn.Module：PyTorch 所有神经网络模块的基类。它覆写了 Python 的 __setattr__。当你给类属性赋值时，如果是 nn.Module 或 nn.Parameter，它会自动将其归入内部字典（_modules、_parameters）。
- nn.Parameter：torch.Tensor 的子类。默认 requires_grad=True。只有挂载为 nn.Parameter 的 Tensor，调用 model.parameters() 时才能被优化器检索到。

nn.Module 是一个“层级状态管理器”，而 nn.Parameter 是给普通 Tensor 贴上“这是网络可学习权重”的身份标识。理解它们的核心，在于弄清楚PyTorch 是如何自动收集、转移和保存参数的。

**为什么普通 torch.Tensor 不够用？**

如果只用基础的 Python 类和 torch.Tensor(..., requires_grad=True)，虽然反向传播求导可以正常计算，但面临三个致命工程问题：

- 优化器拿不到参数：torch.optim.Adam(model.parameters()) 怎么知道网络里一共有哪些 Tensor 需要更新？
- 硬件迁移极度繁琐：执行 model.to("cuda") 时，谁来负责把成百上千个散落的 Tensor 一并搬到 GPU？
- 模型无法统一保存：导出 model.state_dict() 时，如何把所有权重按名字打包？

nn.Module 和 nn.Parameter 配合出现，就是为了解决这三大管理问题。

** nn.Parameter 的本质到底是什么？**

从源码继承关系来看：

- nn.Parameter 继承自 torch.Tensor，它就是一个真正的 Tensor，支持所有张量数学运算。
- 它唯一与普通 Tensor 不同的地方有两点：
    - 默认设置 requires_grad=True（普通 Tensor 默认为 False）。
    - 它是一个特殊标记类。当它被赋值给 nn.Module 的属性时，nn.Module 会立刻将其收编入管。

In [1]:
import torch
import torch.nn as nn

class CustomLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # 正确做法：使用 nn.Parameter 自动注册到 self._parameters
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features))

        # 错误示范：普通 Tensor 不会被 parameters() 跟踪
        self.untracked_tensor = torch.randn(out_features)

    def forward(self, x):
        return x @ self.weight.t() + self.bias

layer = CustomLinear(4, 2)

print("=== 模型识别出的参数 ===")
for name, param in layer.named_parameters():
    print(f"Name: {name}, Shape: {param.shape}, Requires Grad: {param.requires_grad}")
# 输出只包含 weight 和 bias，untracked_tensor 不在其中

=== 模型识别出的参数 ===
Name: weight, Shape: torch.Size([2, 4]), Requires Grad: True
Name: bias, Shape: torch.Size([2]), Requires Grad: True


# forward 方法与生命周期

**核心机制**

在定义模型时必须覆写 forward 方法，但在调用时必须使用 model(x)，绝不能调用 model.forward(x)。

- nn.Module 实现了 __call__ 方法。
- __call__ 内部负责处理：前向前/后 Hook（register_forward_pre_hook / register_forward_hook）、性能分析（Profiler）、以及调用你写的 forward。

In [2]:
import torch
import torch.nn as nn

class HookDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(3, 3)

    def forward(self, x):
        return self.linear(x) * 2

model = HookDemo()

# 注册一个前向 Hook（常用于特征提取或调试）
def hook_fn(module, input, output):
    print(f"[Hook 捕获] 输入形状: {input[0].shape}, 输出形状: {output.shape}")

hook_handle = model.register_forward_hook(hook_fn)
x = torch.randn(1, 3)

print("--- 方式 A: model(x) ---")
out_a = model(x) # 触发 Hook

print("\n--- 方式 B: model.forward(x) [反模式] ---")
out_b = model.forward(x) # 不会触发 Hook

--- 方式 A: model(x) ---
[Hook 捕获] 输入形状: torch.Size([1, 3]), 输出形状: torch.Size([1, 3])

--- 方式 B: model.forward(x) [反模式] ---


In [5]:
import torch
import torch.nn as nn

t = torch.randn(2, 2)
p = nn.Parameter(t)

print("是否为 Tensor 子类:", isinstance(p, torch.Tensor))  # True
print("是否默认跟踪梯度:", p.requires_grad)                   # True

是否为 Tensor 子类: True
是否默认跟踪梯度: True


**nn.Module 是如何“感知”到参数的？（属性拦截机制）nn.Module 是如何“感知”到参数的？（属性拦截机制）**

nn.Module 的底层实现里覆写了 Python 的 __setattr__ 方法。每次你在类内部写 self.xxx = yyy 时，都会触发一次分类拦截：

- 如果右侧是 nn.Parameter：不放进常规的 self.__dict__，而是存入内部字典 self._parameters['xxx'] = yyy。
- 如果右侧是 nn.Module（如 nn.Linear、nn.Conv2d）：存入内部字典 self._modules['xxx'] = yyy。
- 如果是普通 torch.Tensor 或普通 Python 变量（如整数、列表）：直接作为普通属性存入 self.__dict__，PyTorch 完全不会跟踪它。

**三大联动机制在底层如何生效？**

当你把参数注册进 _parameters 后，nn.Module 提供的所有核心能力才能生效：
- model.parameters()：递归遍历自身的 _parameters 以及所有子模块（_modules）的 _parameters，生成一个迭代器返回给优化器。
- model.to(device)：递归找到所有 _parameters 中的 Tensor，原地调用 .to(device)，免去逐个迁移的负担。
- model.state_dict()：提取所有 _parameters 里的权重张量，连同通过 register_buffer 注册的缓存变量，拼接成键值对字典。

In [6]:
import torch
import torch.nn as nn

class TrapDemo(nn.Module):
    def __init__(self):
        super().__init__()
        # 正确注册：单个 Parameter
        self.w1 = nn.Parameter(torch.ones(2, 2))

        # 错误示范 1：普通 Tensor 不会被识别
        self.w2 = torch.ones(2, 2, requires_grad=True)

        # 错误示范 2：原生 Python 列表里的层或参数会“隐身”
        # self.layers 是个普通 list，Module 无法感知列表里面的子层
        self.hidden_layers_bad = [nn.Linear(2, 2), nn.Linear(2, 2)]

        # 正确做法：必须用 nn.ModuleList 或 nn.ParameterList 包装
        self.hidden_layers_good = nn.ModuleList([nn.Linear(2, 2), nn.Linear(2, 2)])

model = TrapDemo()

print("=== 模型成功识别并纳管的参数名 ===")
for name, param in model.named_parameters():
    print(f"参数路径: {name}")

=== 模型成功识别并纳管的参数名 ===
参数路径: w1
参数路径: hidden_layers_good.0.weight
参数路径: hidden_layers_good.0.bias
参数路径: hidden_layers_good.1.weight
参数路径: hidden_layers_good.1.bias


# state_dict 与状态保存

**核心机制**

state_dict 是一个标准 Python 字典，存放两类内容：

- Parameters：需要梯度的可学习参数（权重、偏置）。
- Buffers：模型运行所需的非学习状态（例如 BatchNorm 的 running_mean、running_var）。需要通过 self.register_buffer('name', tensor) 声明。

保存与恢复模型最稳健的方式是保存/加载 state_dict，而不是直接序列化整个模型对象。

state_dict（状态字典）就是 PyTorch 用来抽离、封存和传递“运行状态”的核心数据结构。

**state_dict 的本质到底装了什么？**

state_dict 本质上是一个标准的 Python 字典（早期版本为 collections.OrderedDict）。它的 Key 是参数或缓存的层级路径字符串（如 layer1.conv.weight），Value 则是对应的 torch.Tensor。

state_dict 只包含两类数据：

- Parameters：可学习参数（权重 weight、偏置 bias）。
- Buffers：模型运行所需的持久化统计量或常量（如 BatchNorm 的均值与方差、固定的位置编码）。

| 属性类型           | 声明方式                                               | 是否求导（`requires_grad`） | 是否被优化器感知（`model.parameters()`） | 随 `model.to(device)` 迁移？ | 写入 `state_dict`？ |
| -------------- | -------------------------------------------------- | --------------------- | ------------------------------ | ------------------------ | ---------------- |
| 普通 Python 属性   | `self.x = torch.zeros(1)`                          | 否（默认）                 | 否                              | **否（大坑）**                | 否                |
| `nn.Parameter` | `self.w = nn.Parameter(...)`                       | **是**                 | **是**                          | **是**                    | **是**            |
| 持久化 Buffer     | `self.register_buffer('b', ...)`                   | 否                     | 否                              | **是**                    | **是**            |
| 非持久化 Buffer    | `self.register_buffer('b', ..., persistent=False)` | 否                     | 否                              | **是**                    | 否                |



In [3]:
import torch
import torch.nn as nn

class ModelWithBuffer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(2, 2))
        # 声明 Buffer：不需要梯度，但属于模型状态的一部分
        self.register_buffer('running_counter', torch.zeros(1))

    def forward(self, x):
        self.running_counter += 1
        return x @ self.weight

model = ModelWithBuffer()
dummy_input = torch.randn(1, 2)
model(dummy_input)

# 查看 state_dict
print("state_dict 包含的内容:")
for k, v in model.state_dict().items():
    print(f"  {k}: {v.shape}")

# 保存与加载参数规范
torch.save(model.state_dict(), "model_weights.pt")

new_model = ModelWithBuffer()
new_model.load_state_dict(torch.load("model_weights.pt", weights_only=True))
print("成功加载权重，Counter 值为:", new_model.running_counter.item())

state_dict 包含的内容:
  weight: torch.Size([2, 2])
  running_counter: torch.Size([1])
成功加载权重，Counter 值为: 1.0


# train() 与 eval() 模式切换

**核心机制**

model.train() 和 model.eval() 只是递归地修改每个模块内部的布尔值属性 self.training（True 或 False）。

**主要影响两大算子：**

nn.Dropout：

- train()：以概率 $p$ 将元素置 0，其余元素缩放 $\frac{1}{1-p}$。
- eval()：直接恒等映射（Identity），不丢弃神经元。

nn.BatchNorm：

- train()：利用当前 Batch 计算均值和方差，并更新 running_mean 与 running_var。
- eval()：停止更新统计量，直接使用保存下来的 running_mean 和 running_var 归一化。

> 关键澄清：model.eval() 绝对不会关闭反向传播梯度计算。推理时必须显式搭配 torch.no_grad() 才能节省显存与算力。

In [4]:
import torch
import torch.nn as nn

class BehaviorDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(p=0.5)
        self.linear = nn.Linear(4, 2)

    def forward(self, x):
        return self.linear(self.dropout(x))

model = BehaviorDemo()
x = torch.ones(2, 4)

# 1. 训练模式
model.train()
print(f"model.training 状态: {model.training}")
print("Train 输出 (Dropout 生效，有置 0):")
print(model.dropout(x))

# 2. 评估模式
model.eval()
print(f"\nmodel.training 状态: {model.training}")
print("Eval 输出 (Dropout 失效，原样保留):")
print(model.dropout(x))

# 3. 验证梯度陷阱
out = model(x)
loss = out.sum()
loss.backward()
print("\neval 模式下仍然计算了梯度:", model.linear.weight.grad is not None)
# 正确评估做法：
# with torch.no_grad():
#     out = model(x)

model.training 状态: True
Train 输出 (Dropout 生效，有置 0):
tensor([[2., 2., 2., 2.],
        [0., 2., 2., 0.]])

model.training 状态: False
Eval 输出 (Dropout 失效，原样保留):
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])

eval 模式下仍然计算了梯度: True


## 两大核心算子的数学与行为差异

### nn.Dropout：Inverted Dropout（反向丢弃）

标准实现并非在测试时做缩放，而是在训练时提前完成缩放（Inverted Dropout），从而保证推理时完全为零开销的恒等映射。

train() 模式：

以概率 $p$ 将元素置 0，并对保留的元素除以 $(1-p)$：

$$y = \frac{1}{1-p} \cdot m \odot x, \quad m \sim \text{Bernoulli}(1-p)$$

除以 $(1-p)$ 是为了保证在期望层面上，输出的激活值均值保持一致：

$$\mathbb{E}[y] = \frac{1}{1-p} \cdot (1-p) \cdot x = x$$

### eval() 模式：

直接作为恒等映射（Identity）：
$$y = x$$
所有神经元均激活，不做任何置 0 或缩放处理。

### nn.BatchNorm2d：统计量的切换与维护

BatchNorm 是模式切换中逻辑最复杂的算子，其在两个模式下的均值和方差来源完全不同。

train() 模式：

计算当前 mini-batch 的均值 $\mu_B$ 与方差 $\sigma_B^2$：

$$\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i, \quad \sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2$$

使用当前 Batch 的统计量对输入进行归一化：

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

通过指数移动平均（EMA）更新全局缓存统计量（Buffers: running_mean, running_var）：

$$\text{running\_mean} \leftarrow (1 - \text{momentum}) \times \text{running\_mean} + \text{momentum} \times \mu_B$$

$$\text{running\_var} \leftarrow (1 - \text{momentum}) \times \text{running\_var} + \text{momentum} \times \sigma_{B, \text{unbiased}}^2$$

### eval() 模式：

完全停止更新 running_mean 与 running_var。

归一化时不再计算当前 Batch 的均值和方差，而是直接使用训练期间固化下来的 Buffer：

$$\hat{x}_i = \frac{x_i - \text{running\_mean}}{\sqrt{\text{running\_var} + \epsilon}}$$

> 致命陷阱：如果在推理时忘记调用 model.eval() 且传入 batch_size = 1 的数据，BatchNorm 在 train() 模式下计算单个样本的方差为 0，除以接近 0 的值会导致数值爆炸，直接输出 NaN 或全 0。

